In [1]:
import heapq

In [2]:
class PriorityQueue:
    def __init__(self):
        self.__data = []

    # Función para verificar si la fila de prioridades está vacía
    def empty(self):
        return not self.__data

    # Función para limpiar la fila de prioridades
    def clear(self):
        self.__data.clear()
    
    # Función para insertar un elemento en la fila de prioridades
    def push(self, priority, value):
        heapq.heappush(self.__data, (priority, value))

    # Función para extraer el elemento con mayor prioridad (menor número)
    def pop(self):
        if self.__data: # not empty
            heapq.heappop(self.__data)
        else:
            raise Exception("No such element")
        
    # Función para obtener el primer elemento sin sacarlo
    def top(self):
        if self.__data: # not empty
            return self.__data[0]
        else:
            raise Exception("No such element")

In [3]:
pq = PriorityQueue()

# Insertar elementos en la fila de prioridades
pq.push(4, 'task4')
pq.push(2, 'task2')
pq.push(1, 'task1')
pq.push(5, 'task5')
pq.push(3, 'task3')

# Verificar si la fila de prioridades está vacía
print(f'¿La fila de prioridades está vacía?: {pq.empty()}')

# Obtener el primer elemento sin sacarlo
first_element = pq.top()
print(f'El primer elemento es: Prioridad: {first_element[0]}, Valor: {first_element[1]}')

# Extraer e imprimir elementos según su prioridad
while not pq.empty():
    (priority, value) = pq.top()
    pq.pop()
    print(f'Prioridad: {priority}, Valor: {value}')

# Limpiar la fila de prioridades
pq.clear()
print(f'¿La fila de prioridades está vacía después de limpiar?: {pq.empty()}')

¿La fila de prioridades está vacía?: False
El primer elemento es: Prioridad: 1, Valor: task1
Prioridad: 1, Valor: task1
Prioridad: 2, Valor: task2
Prioridad: 3, Valor: task3
Prioridad: 4, Valor: task4
Prioridad: 5, Valor: task5
¿La fila de prioridades está vacía después de limpiar?: True


# Contexto: 
Imagina que estás en una ciudad grande y quieres encontrar la ruta más rápida para llegar desde tu casa (0,0) a una popular cafetería en el otro extremo de la ciudad (5,7). Usarás dos enfoques diferentes (Dijkstra y A*) para encontrar la mejor ruta.

In [4]:
INFINITE = 1_000_000
matrix = [
    [3, 9, 5, 6, 4, -1, 2, 10],
    [1, 7, 8, 4, 3, 5, 9, 2],
    [4, -1, 7, 3, 6, 2, 10, 8],
    [5, 3, 2, 9, 7, 1, 6, 4],
    [8, 6, 1, 7, 4, 10, 5, -1],
    [2, 4, 6, 9, 3, 8, 4, 7],
]

In [5]:
def to_int(matrix, position):
    (row, col) = position
    rows = len(matrix)
    cols = len(matrix[0])
    return (row * cols) + col

def to_coord(matrix, i):
    rows = len(matrix)
    cols = len(matrix[0])

    return ( (i // rows), (i % cols) )

In [6]:
print(to_int(matrix, (1, 2)))
print(to_coord(matrix, 10))

10
(1, 2)


In [14]:
def is_valid(matrix, position):
    (row, col) = position
    rows = len(matrix)
    cols = len(matrix[0])
    return 0 <= row < rows and 0 <= col < cols

def get_neighborhood(matrix, position):
    result = []
    
    (ren, col) = position
    
    new_position = ((ren - 1), col)
    if is_valid(matrix, new_position):
        result.append(new_position)

    new_position = ((ren + 1), col)
    if is_valid(matrix, new_position):
        result.append(new_position)

    new_position = (ren, (col - 1))
    if is_valid(matrix, new_position):
        result.append(new_position)

    new_position = (ren, (col + 1))
    if is_valid(matrix, new_position):
        result.append(new_position)
        
    return result

In [15]:
print(get_neighborhood(matrix, (4, 5)))

[(3, 5), (5, 5), (4, 4), (4, 6)]


## Algoritmo de Dijkstra: 
Piensa en Dijkstra como si fueras un turista curioso que quiere explorar todas las posibles rutas sin tener prisa por llegar al destino.
* **Cómo funciona:** Empiezas desde tu casa y visitas cada calle adyacente una por una, anotando la distancia a cada intersección (nodo). Siempre eliges la intersección más cercana que aún no has visitado y repites este proceso hasta que llegas a la cafetería.
* **En resumen:** Exploras sistemáticamente cada posible ruta en todas direcciones hasta encontrar la más corta a tu destino, sin preocuparte de qué tanto te desviaste en el camino.

In [29]:
def dijkstra(matrix, src, dest):
    n = len(matrix) * len(matrix[0])
    dist = [INFINITE] * n
    prev = [None] * n
    pq = PriorityQueue()
    steps = 0

    dist[to_int(matrix, src)] = 0
    pq.push(0, src)  
    while not pq.empty():
        current_dist, u = pq.top()
        pq.pop()

        # Si hemos llegado al destino, podemos salir del ciclo
        if u == dest:
            break

        for v in get_neighborhood(matrix, u):
            (row, col) = v
            if matrix[row][col] != -1:  
                new_dist = dist[to_int(matrix, u)] + matrix[row][col]

                if new_dist < dist[to_int(matrix, v)]:
                    dist[to_int(matrix, v)] = new_dist
                    prev[to_int(matrix, v)] = u
                    pq.push(new_dist, v)

        steps += 1
    
    # Reconstruir el camino
    path = []
    u = dest
    if prev[to_int(matrix, u)] is not None or u == src:
        while u is not None:
            path.insert(0, u)
            u = prev[to_int(matrix, u)]
    
    return dist[to_int(matrix, dest)], path, steps

In [30]:
src_node = (0, 0)
dest_node = (5, 7)

distance, path, steps = dijkstra(matrix, src_node, dest_node)
print(f"Distancia desde el nodo {src_node} al nodo {dest_node}: {distance}")
print(f"Camino: {path}")
print(f"Pasos: {steps}")

Distancia desde el nodo (0, 0) al nodo (5, 7): 49
Camino: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (4, 2), (4, 3), (4, 4), (5, 4), (5, 5), (5, 6), (5, 7)]
Pasos: 44


## Algoritmo A*:
Piensa en A* como un local que conoce bien la ciudad y tiene un buen sentido de orientación, queriendo llegar lo más rápido posible.
* **Cómo funciona:** Empiezas desde tu casa y no solo consideras la distancia recorrida hasta ahora, sino también tienes en cuenta una especie de "intuición" sobre qué tan cerca estás de la cafetería (heurística). Utilizas esta intuición para priorizar las calles que parecen acercarte más rápidamente a tu destino.
* **En resumen:** A* usa un enfoque más dirigido, combinando la distancia recorrida y una estimación de la distancia restante, lo que te lleva de manera más eficiente a tu destino.

In [31]:
def heuristics(src, dest):
    # Implementa una heurística aquí. Aquí usamos una heurística simple 
    # (distancia Manhattan)
    return (abs(src[0] - dest[0]) + abs(src[0] - dest[0])) * 5

In [32]:
def a_star(matrix, src, dest):
    n = len(matrix) * len(matrix[0])
    dist = [INFINITE] * n
    prev = [None] * n
    pq = PriorityQueue()
    steps = 0

    dist[to_int(matrix, src)] = 0
    pq.push(0, src) 
    while not pq.empty():
        current_dist, u = pq.top()
        pq.pop()

        # Si hemos llegado al destino, podemos salir del ciclo
        if u == dest:
            break

        for v in get_neighborhood(matrix, u):
            (row, col) = v
            if matrix[row][col] != -1:  
                new_dist = dist[to_int(matrix, u)] + matrix[row][col]

                if new_dist < dist[to_int(matrix, v)]:
                    dist[to_int(matrix, v)] = new_dist
                    prev[to_int(matrix, v)] = u
                    priority = new_dist + heuristics(v, dest)
                    pq.push(priority, v)
                    
        steps += 1
    
    # Reconstruir el camino
    path = []
    u = dest
    if prev[to_int(matrix, u)] is not None or u == src:
        while u is not None:
            path.insert(0, u)
            u = prev[to_int(matrix, u)]
    
    return dist[to_int(matrix, dest)], path, steps

In [33]:
src_node = (0, 0)
dest_node = (5, 7)

distance, path, steps = a_star(matrix, src_node, dest_node)
print(f"Distancia desde el nodo {src_node} al nodo {dest_node}: {distance}")
print(f"Camino: {path}")
print(f"Pasos: {steps}")

Distancia desde el nodo (0, 0) al nodo (5, 7): 49
Camino: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (4, 2), (4, 3), (4, 4), (5, 4), (5, 5), (5, 6), (5, 7)]
Pasos: 34


## Conclusión: 
El algoritmo de Dijkstra asegura explorar todas las rutas de manera exhaustiva, lo cual puede ser más lento pero garantiza encontrar la ruta más corta. A* es más eficiente al usar una guía (heurística) que te lleva más directamente al objetivo, a menudo visitando menos nodos.